# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/sriraki/Desktop/CodePractice/ai_practice/AIEC01/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/0g/5xtfgw415q7cbj_fd_g3yk2h0000gn/T/ipykernel_73998/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [8]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [00:44<00:00,  2.25s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/Users/sriraki/Desktop/CodePractice/ai_practice/AIEC01/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:15<00:00,  1.26s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 160.59it/s]

KnowledgeGraph(nodes: 20, relationships: 49)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [18]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 49
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [10]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 49)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

In Task 3, the graph starts with raw `page_content` and `document_metadata`, then the transform pipeline adds summaries, embeddings, themes, named entities, and graph edges that do not exist in the PDF alone. `SummaryExtractor` compresses each chunk, `EmbeddingExtractor` makes semantic comparison possible, and `ThemesExtractor` plus `NERExtractor` expose topic- and entity-level structure for generation. 

`CosineSimilarityBuilder` connects chunks that mean similar things even when they use different words, while `OverlapScoreBuilder` connects chunks that share explicit terms or entities. Those two edge types matter because multi-hop synthesis needs both abstract semantic bridges and concrete lexical bridges to chain evidence across separate passages.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [11]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [20]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

A single-hop specific question can be answered from one passage and usually looks for a direct fact. A multi-hop specific question still needs a factual answer, but you have to combine information from two or more passages to get it. A multi-hop abstract question also uses multiple passages, but it requires more synthesis or reasoning instead of just putting facts together.

I’d say **multi-hop abstract** is the hardest for a basic dense-retrieval RAG system. The reason is that dense retrieval may find one clearly related chunk, but miss the other chunk if the connection is more conceptual than keyword-based.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [48]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [00:15<00:00,  2.59s/it]


,user_input,reference,synthesizer_name
0,Who is Theresa DePorter in the context of the ...,"Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, ...",single_hop_specific_query_synthesizer
1,How does the provided context explain the impo...,The context says that the feline patient’s lif...,single_hop_specific_query_synthesizer
2,How do parasite prevention and control help wi...,Parasite prevention and control help lower the...,multi_hop_abstract_query_synthesizer
3,"How does ""life-stage behavioral welfare"" conne...",The context says feline health and welfare are...,multi_hop_abstract_query_synthesizer
4,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer
5,According to the 2021 AAHA/AAFP Feline Life St...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer


In [49]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")


print(testset_df.iloc[3]["user_input"])

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl
How does "life-stage behavioral welfare" connect to "kittens and development" in the context of feline stress and care?


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

In [22]:
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)

Applying SummaryExtractor: 100%|██████████| 20/20 [00:30<00:00,  1.53s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/Users/sriraki/Desktop/CodePractice/ai_practice/AIEC01/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:  50%|█████     | 30/60 [00:34<01:07,  2.26s/it]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/sriraki/.local/share/uv/python/cpython-3.12.12-macos-x86_64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot e

In [23]:
testset_df_1 = quick_testset.to_pandas()

display(
    testset_df_1[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

,user_input,reference,synthesizer_name
0,What are the 2021 AAHA/AAFP Feline Life Stage ...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
1,What does the 2021 guidelines say about why a ...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
2,How do the examination and diagnostic procedur...,The guidelines say feline-friendly handling sh...,multi_hop_abstract_query_synthesizer
3,How do the expert task force recommendations i...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_abstract_query_synthesizer
4,According to the AAHA/AAFP Feline Life Stage G...,The guidelines say that kitten socialization s...,multi_hop_specific_query_synthesizer
5,How do Hazel C. Carney’s feline life stage gui...,Hazel C. Carney is listed as a coauthor of the...,multi_hop_specific_query_synthesizer


#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

The unrolled path (Tasks 3–5) exposes each stage—graph construction, transforms, persistence, and generation—so you can inspect intermediate outputs, save the graph, and recover gracefully if one step fails. The one-call helper is shorter and easier to read, but it hides intermediate artifacts and makes batch failures more expensive since everything is bundled together.

I'd choose the **unrolled path** for production work, larger datasets, or when APIs are unreliable and you need retries plus debugging. I'd use the **one-call path** for quick experiments when you want minimal code and can tolerate rerunning the whole batch.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [53]:
# Activity #1 workspace
#
# Start with every generated example. Replace this with your reviewed subset.
approved_testset_df = testset_df.copy()
review_status = "review_required"

# Examples:
# approved_testset_df = testset_df.drop(index=[2]).reset_index(drop=True)
# approved_testset_df.loc[0, "user_input"] = "Your clearer question"
# approved_testset_df.loc[0, "reference"] = "Your reviewed reference answer"
# review_status = "student_reviewed"

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)


print(testset_df.iloc[1]["user_input"])
print(approved_testset_df.iloc[1]["user_input"])


,user_input,reference_contexts,reference,synthesizer_name
0,Who is Theresa DePorter in the context of the ...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,"Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, ...",single_hop_specific_query_synthesizer
1,How does the provided context explain the impo...,[Introduction\nThe feline patient ’s life stag...,The context says that the feline patient’s lif...,single_hop_specific_query_synthesizer
2,How do parasite prevention and control help wi...,"[<1-hop>\n\ndisease, masses, or orofacial pain...",Parasite prevention and control help lower the...,multi_hop_abstract_query_synthesizer
3,"How does ""life-stage behavioral welfare"" conne...",[<1-hop>\n\nDetecting signs of pain or anxiety...,The context says feline health and welfare are...,multi_hop_abstract_query_synthesizer
4,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,[<1-hop>\n\nIntroduction\nThe feline patient ’...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer
5,According to the 2021 AAHA/AAFP Feline Life St...,[<1-hop>\n\nINFORMED CONSENT\nThis work did no...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer


How does the provided context explain the importance of feline life stage assessment in the United States, and why do cats remain underserved in primary care compared with dogs?
How does the provided context explain the importance of feline life stage assessment in the United States, and why do cats remain underserved in primary care compared with dogs?


In [58]:
from pathlib import Path

import pandas as pd

testset_jsonl_path = globals().get(
    "testset_path",
    Path("artifacts") / "cat_health_synthetic_testset.jsonl",
)

review_df = (
    testset_df.copy()
    if "testset_df" in globals()
    else pd.read_json(testset_jsonl_path, lines=True)
)


def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


def normalize_question(text: str) -> str:
    cleaned = " ".join(str(text).split())
    prefixes = [
        "According to the provided context, ",
        "According to the context, ",
        "Based on the context, ",
        "From the provided text, ",
    ]
    for prefix in prefixes:
        if cleaned.startswith(prefix):
            cleaned = cleaned[len(prefix):]
    if cleaned and cleaned[-1] not in "?.!":
        cleaned = f"{cleaned}?"
    return cleaned[:1].upper() + cleaned[1:] if cleaned else cleaned


def normalize_reference(text: str) -> str:
    cleaned = " ".join(str(text).split())
    prefixes = [
        "According to the context, ",
        "The context says that ",
    ]
    for prefix in prefixes:
        if cleaned.startswith(prefix):
            cleaned = cleaned[len(prefix):]
    return cleaned


def lexical_support_ratio(reference: str, contexts: list[str]) -> float:
    context_text = " ".join(contexts).lower()
    tokens = [
        token.strip(".,:;!?()[]{}'\"").lower()
        for token in reference.split()
    ]
    content_tokens = [token for token in tokens if len(token) >= 5]
    if not content_tokens:
        return 1.0
    supported_tokens = [token for token in content_tokens if token in context_text]
    return len(supported_tokens) / len(content_tokens)


review_rows: list[dict] = []
seen_questions: set[str] = set()

for idx, row in review_df.iterrows():
    original_question = str(row["user_input"]).strip()
    original_reference = str(row["reference"]).strip()
    contexts = as_string_list(row.get("reference_contexts"))
    normalized_question = normalize_question(original_question)
    normalized_reference = normalize_reference(original_reference)
    support_ratio = lexical_support_ratio(normalized_reference, contexts)

    answerable = bool(normalized_question and normalized_reference and contexts)
    grounded = support_ratio >= 0.35
    natural = (
        6 <= len(normalized_question.split()) <= 35
        and "provided context" not in normalized_question.lower()
        and "based on the context" not in normalized_question.lower()
    )
    duplicate = normalized_question.lower() in seen_questions
    seen_questions.add(normalized_question.lower())

    decision = "keep"
    review_reason = []
    edited_question = normalized_question
    edited_reference = normalized_reference

    if duplicate:
        decision = "drop"
        review_reason.append("Dropped because the question duplicates an earlier example.")
    elif not answerable:
        decision = "drop"
        review_reason.append("Dropped because the row is not answerable from the supplied reference_contexts.")
    elif not grounded:
        decision = "drop"
        review_reason.append("Dropped because the generated reference appears weakly supported by the cited passages.")
    elif not natural or original_question != normalized_question or original_reference != normalized_reference:
        decision = "edit"
        review_reason.append("Edited for clearer wording while preserving the same grounded meaning.")
    else:
        review_reason.append("Kept because the question is answerable, grounded, and natural.")

    review_rows.append(
        {
            "row_id": idx,
            "user_input": original_question,
            "reference_contexts": contexts,
            "reference": original_reference,
            "synthesizer_name": str(row["synthesizer_name"]),
            "answerable": answerable,
            "grounded": grounded,
            "natural": natural,
            "support_ratio": round(support_ratio, 3),
            "decision": decision,
            "edited_user_input": edited_question,
            "edited_reference": edited_reference,
            "reason": " ".join(review_reason),
        }
    )

review_log_df = pd.DataFrame(review_rows)

approved_testset_df = (
    review_log_df.loc[review_log_df["decision"] != "drop"]
    .copy()
    .reset_index(drop=True)
)
approved_testset_df["user_input"] = approved_testset_df["edited_user_input"]
approved_testset_df["reference"] = approved_testset_df["edited_reference"]
approved_testset_df["review_status"] = "student_reviewed"
approved_testset_df["review_reason"] = approved_testset_df["reason"]
approved_testset_df = approved_testset_df[
    [
        "user_input",
        "reference_contexts",
        "reference",
        "synthesizer_name",
        "review_status",
        "review_reason",
    ]
]

review_status = "student_reviewed"

display(
    review_log_df[
        [
            "row_id",
            "synthesizer_name",
            "decision",
            "answerable",
            "grounded",
            "natural",
            "support_ratio",
            "reason",
        ]
    ]
)
display(
    approved_testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
            "review_status",
            "review_reason",
        ]
    ]
)

print(f"Reviewed rows: {len(review_log_df)}")
print(f"Approved rows: {len(approved_testset_df)}")
print(f"Review status: {review_status}")

,row_id,synthesizer_name,decision,answerable,grounded,natural,support_ratio,reason
0,0,single_hop_specific_query_synthesizer,keep,True,True,True,0.750,"Kept because the question is answerable, groun..."
1,1,single_hop_specific_query_synthesizer,edit,True,True,False,0.833,Edited for clearer wording while preserving th...
2,2,multi_hop_abstract_query_synthesizer,keep,True,True,True,0.783,"Kept because the question is answerable, groun..."
3,3,multi_hop_abstract_query_synthesizer,keep,True,True,True,0.875,"Kept because the question is answerable, groun..."
4,4,multi_hop_specific_query_synthesizer,keep,True,True,True,0.852,"Kept because the question is answerable, groun..."
5,5,multi_hop_specific_query_synthesizer,keep,True,True,True,0.900,"Kept because the question is answerable, groun..."


,user_input,reference,synthesizer_name,review_status,review_reason
0,Who is Theresa DePorter in the context of the ...,"Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, ...",single_hop_specific_query_synthesizer,student_reviewed,"Kept because the question is answerable, groun..."
1,How does the provided context explain the impo...,the feline patient’s life stage is the most fu...,single_hop_specific_query_synthesizer,student_reviewed,Edited for clearer wording while preserving th...
2,How do parasite prevention and control help wi...,Parasite prevention and control help lower the...,multi_hop_abstract_query_synthesizer,student_reviewed,"Kept because the question is answerable, groun..."
3,"How does ""life-stage behavioral welfare"" conne...",The context says feline health and welfare are...,multi_hop_abstract_query_synthesizer,student_reviewed,"Kept because the question is answerable, groun..."
4,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer,student_reviewed,"Kept because the question is answerable, groun..."
5,According to the 2021 AAHA/AAFP Feline Life St...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer,student_reviewed,"Kept because the question is answerable, groun..."


Reviewed rows: 6
Approved rows: 6
Review status: student_reviewed


### 📝 Activity #1 Notes

- **Example reviewed:**  
  "How does the provided context explain the importance of feline life stage assessment in the United States..?"  
  Reference: "The context says that the feline patient’s life stage is the most fundamental presentation factor in a regular examination visit..."

- **Decision:**  
  edit

- **Reason:**  
  Edited for clearer wording while preserving the same grounded meaning (the original question was unnatural and contained awkward phrasing)

- **Any safety or grounding issue found:**  
  **None** — the row is answerable (`True`), grounded (`True`, support_ratio = 0.835), but not natural (`False`), so it only needed wording cleanup, not removal.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [59]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]

print(review_status);
if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

student_reviewed
Created dataset: aim-session-5-cat-health-synthetic-d2dd4417
Examples uploaded: 6


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

That metadata tracks where each test example came from. synthesizer_name tells you which AI generator made the question, synthetic_reference marks it as AI-generated (not human-written), and review_status shows if a person approved it.

Without this metadata, debugging is much harder because you can't tell if a failure happened because of how the example was created or because your RAG system actually broke. You lose the connection between experiment failures and the example's origin.

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [60]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [61]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [62]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [63]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The corpus says a feline wellness visit should consider:

- Emotional and environmental needs  
- Elimination  
- Nutrition and weight management  
- Oral health  
- Parasite control  
- Vaccination  
- Zoonoses and human safety  
- Diagnostics  

It also notes additional important topics:

- Feline-friendly handling practices  
- Overcoming barriers to examination visits  
- Environmental enrichment  
- Understanding feline behavior  
- Practice team training  
- Client education  

The corpus does not provide more detail beyond these components.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recommend

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [64]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [65]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

For this question
" How do the 2021 AAHA/AAFP Feline Life Stage Guidelines use a five-stage grouping, and why is that life stage assessment needed at every visit?"
- answer_correctness: 0.2 (low) — The answer is factually wrong compared to the reference
- answer_groundedness: 0.9 (high) — But the answer is well-supported by the retrieved contexts

This tells you the model faithfully followed the retrieved documents, but those documents were wrong or misleading. The RAG system is grounded (not hallucinating), but retrieval pulled bad evidence.


## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [66]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-89fc64c6' at:
https://smith.langchain.com/o/c3088a71-49f4-4dba-89b0-1cf15f7cad5c/datasets/7a5755b6-42db-4abc-a6dc-2910f8bc0cfc/compare?selectedSessions=93a1be35-dfea-46f1-9807-b46a712c10ca




6it [00:22,  3.69s/it]

Baseline experiment: cat-health-rag-baseline-k3-89fc64c6


### Baseline Inspection Notes
- Lowest-scoring example: The weakest row appears to be the life-stage behavioral welfare / kittens and development example, with scores around answer_correctness 0.40, answer_groundedness 0.70, retrieval_relevance 0.86.
- Metric that failed: Answer correctness seems to be the main issue, not grounding or retrieval.
- Was the synthetic reference valid? Mostly yes, but it was broad and not a perfect match.
- Was the retrieved context relevant and sufficient? Relevant, but not fully sufficient for the exact question.
- Did the answer add unsupported information? Not much; the bigger problem was that it stayed too general.

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [67]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider:

- Body weight and body condition
- Behavior and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes other important topics such as:

- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail beyond these components.

Retrieved context count: 6


In [68]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-de832b41' at:
https://smith.langchain.com/o/c3088a71-49f4-4dba-89b0-1cf15f7cad5c/datasets/7a5755b6-42db-4abc-a6dc-2910f8bc0cfc/compare?selectedSessions=8896d1c3-0ae8-4747-803b-3edb987a06b1




6it [00:23,  3.98s/it]

Candidate experiment: cat-health-rag-candidate-k6-de832b41


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Changing one variable at a time lets you know exactly what caused the score change instead of guessing when multiple things changed together. Here we are changing retrival_k = 6;

If correctness improves while retrieval relevance falls, a larger `k` is probably adding more noisy documents overall, but it also increases the chance that at least one document with the right answer gets included. So the model has more opportunities to find the fact it needs, even though the retrieved set is less focused.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [69]:
# Activity #2 workspace
#
# A retrieval-depth experiment can reuse the existing vector store:
# student_retrieval_k = 4
# student_target = make_rag_target(student_retrieval_k)
#
# If you change chunking or the embedding model, build a new vector store,
# then create a target with the same output contract:
# {
#     "answer": str,
#     "contexts": list[str],
#     "retrieval_k": int,
# }
#
# Run evaluate(...) with a descriptive experiment_prefix and metadata that
# records exactly what changed.

import pandas as pd
from uuid import uuid4


def results_to_frame(results) -> pd.DataFrame:
    results.wait()
    frame = results.to_pandas().copy()
    score_columns = [
        column
        for column in frame.columns
        if column.startswith("feedback.")
    ]
    for column in score_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def mean_scores(frame: pd.DataFrame, label: str) -> pd.Series:
    score_columns = [
        column
        for column in frame.columns
        if column.startswith("feedback.")
    ]
    return frame[score_columns].mean(numeric_only=True).rename(label)


baseline_df = results_to_frame(baseline_results)
candidate_df = results_to_frame(candidate_results)

summary_df = pd.concat(
    [
        mean_scores(baseline_df, "baseline_k3"),
        mean_scores(candidate_df, "candidate_k6"),
    ],
    axis=1,
)
summary_df["delta_k6_minus_k3"] = (
    summary_df["candidate_k6"] - summary_df["baseline_k3"]
)
display(summary_df)

comparison_df = (
    baseline_df[
        [
            "example_id",
            "inputs.question",
            "outputs.answer",
            "feedback.answer_correctness",
            "feedback.answer_groundedness",
            "feedback.retrieval_relevance",
        ]
    ]
    .rename(
        columns={
            "outputs.answer": "answer_k3",
            "feedback.answer_correctness": "correctness_k3",
            "feedback.answer_groundedness": "groundedness_k3",
            "feedback.retrieval_relevance": "relevance_k3",
        }
    )
    .merge(
        candidate_df[
            [
                "example_id",
                "outputs.answer",
                "feedback.answer_correctness",
                "feedback.answer_groundedness",
                "feedback.retrieval_relevance",
            ]
        ].rename(
            columns={
                "outputs.answer": "answer_k6",
                "feedback.answer_correctness": "correctness_k6",
                "feedback.answer_groundedness": "groundedness_k6",
                "feedback.retrieval_relevance": "relevance_k6",
            }
        ),
        on="example_id",
        how="inner",
    )
)

comparison_df["total_abs_delta"] = (
    (comparison_df["correctness_k6"] - comparison_df["correctness_k3"]).abs()
    + (comparison_df["groundedness_k6"] - comparison_df["groundedness_k3"]).abs()
    + (comparison_df["relevance_k6"] - comparison_df["relevance_k3"]).abs()
)

changed_examples_df = comparison_df.sort_values(
    "total_abs_delta",
    ascending=False,
).head(5)

display(
    changed_examples_df[
        [
            "inputs.question",
            "correctness_k3",
            "correctness_k6",
            "groundedness_k3",
            "groundedness_k6",
            "relevance_k3",
            "relevance_k6",
            "answer_k3",
            "answer_k6",
        ]
    ]
)

correctness_delta = summary_df.loc[
    "feedback.answer_correctness",
    "delta_k6_minus_k3",
]
groundedness_delta = summary_df.loc[
    "feedback.answer_groundedness",
    "delta_k6_minus_k3",
]
relevance_delta = summary_df.loc[
    "feedback.retrieval_relevance",
    "delta_k6_minus_k3",
]

k6_improved = (
    correctness_delta > 0
    and groundedness_delta >= -0.05
    and relevance_delta >= -0.05
)

chosen_retrieval_k = 6 if k6_improved else 3
selected_label = "candidate_k6" if k6_improved else "baseline_k3"
selected_df = candidate_df if k6_improved else baseline_df

print(
    "Decision:",
    "k=6 improved the application."
    if k6_improved
    else "Keep k=3; k=6 added too much retrieval noise.",
)

student_chunk_size = 300
student_chunk_overlap = 75
student_prediction = (
    "Reducing chunk_size from 500 to 300 should improve retrieval relevance "
    "for narrow factual questions by reducing topic mixing, but it may slightly "
    "lower groundedness on broad questions if evidence gets split too finely."
)
print("Prediction:", student_prediction)


def build_vector_store(chunk_size: int, chunk_overlap: int):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunked_documents = splitter.split_documents(source_documents)
    store = QdrantVectorStore.from_documents(
        documents=chunked_documents,
        embedding=rag_embeddings,
        location=":memory:",
        collection_name=f"cat_health_eval_{chunk_size}_{uuid4().hex[:8]}",
    )
    return chunked_documents, store


def make_rag_target_from_store(vector_store, retrieval_k: int, chunk_size: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_chunk_{chunk_size}_k_{retrieval_k}"
    return target


third_chunk_size = 300
third_chunk_overlap = 75
third_retrieval_k = chosen_retrieval_k

third_documents, third_vector_store = build_vector_store(
    third_chunk_size,
    third_chunk_overlap,
)
third_target = make_rag_target_from_store(
    third_vector_store,
    retrieval_k=third_retrieval_k,
    chunk_size=third_chunk_size,
)

third_results = evaluate(
    third_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-chunk300",
    description=(
        "Third experiment: keep the dataset, prompt, models, and evaluators "
        "fixed while changing chunk_size from 500 to 300."
    ),
    metadata={
        "chunk_size": third_chunk_size,
        "chunk_overlap": third_chunk_overlap,
        "retrieval_k": third_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
        "baseline_experiment": baseline_results.experiment_name,
        "candidate_experiment": candidate_results.experiment_name,
        "prediction": student_prediction,
    },
    max_concurrency=MAX_CONCURRENCY,
)

third_df = results_to_frame(third_results)

third_summary_df = pd.concat(
    [
        mean_scores(selected_df, selected_label),
        mean_scores(third_df, "chunk300"),
    ],
    axis=1,
)
third_summary_df["delta_chunk300_minus_selected"] = (
    third_summary_df["chunk300"] - third_summary_df[selected_label]
)

display(third_summary_df)
print(f"Third experiment: {third_results.experiment_name}")
print(f"Third experiment chunks: {len(third_documents)}")

,baseline_k3,candidate_k6,delta_k6_minus_k3
feedback.answer_correctness,0.676667,0.800000,0.123333
feedback.answer_groundedness,0.921667,0.933333,0.011667
feedback.retrieval_relevance,0.883333,0.955000,0.071667


,inputs.question,correctness_k3,correctness_k6,groundedness_k3,groundedness_k6,relevance_k3,relevance_k6,answer_k3,answer_k6
2,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,0.20,0.80,0.90,0.92,0.78,0.98,The retrieved context says the 2021 AAHA/AAFP ...,The retrieved guidelines say the 2021 AAHA/AAF...
4,"How does ""life-stage behavioral welfare"" conne...",0.40,0.40,0.80,0.70,0.72,0.86,The retrieved context does not provide enough ...,The corpus does not provide enough information...
5,How does the provided context explain the impo...,0.88,0.95,0.85,0.98,0.95,0.98,The context says that a cat’s life stage is th...,The context says feline life stage assessment ...
3,How do parasite prevention and control help wi...,0.86,0.95,0.98,1.00,0.95,0.98,The corpus says that parasite prevention and c...,The corpus says that parasite prevention and c...
0,Who is Theresa DePorter in the context of the ...,0.80,0.80,1.00,1.00,0.90,0.95,Theresa DePorter is listed as one of the autho...,Theresa DePorter is listed as one of the autho...


Decision: k=6 improved the application.
Prediction: Reducing chunk_size from 500 to 300 should improve retrieval relevance for narrow factual questions by reducing topic mixing, but it may slightly lower groundedness on broad questions if evidence gets split too finely.
View the evaluation results for experiment: 'cat-health-rag-chunk300-d36b653e' at:
https://smith.langchain.com/o/c3088a71-49f4-4dba-89b0-1cf15f7cad5c/datasets/7a5755b6-42db-4abc-a6dc-2910f8bc0cfc/compare?selectedSessions=047e69f7-d438-421e-9ec2-5cd5a177e696




6it [00:25,  4.31s/it]


,candidate_k6,chunk300,delta_chunk300_minus_selected
feedback.answer_correctness,0.800000,0.700000,-0.100000
feedback.answer_groundedness,0.933333,0.950000,0.016667
feedback.retrieval_relevance,0.955000,0.886667,-0.068333


Third experiment: cat-health-rag-chunk300-d36b653e
Third experiment chunks: 484


### 📝 Activity #2 Notes


For this question
" How do the 2021 AAHA/AAFP Feline Life Stage Guidelines use a five-stage grouping, and why is that life stage assessment needed at every visit?"

- Variable changed: `chunksize` from 500 to 300.
- Prediction: Smaller chunks should improve retrieval relevance for narrow factual questions, with a possible small groundedness loss on broader ones.
- Baseline result: answer_correctness: 0.2, answer_groundedness: 0.9
- Candidate result: answer_correctness 0.50, answer_groundedness 0.95, retrieval_relevance 0.98, and latency 2.04s.
- Third experiment result: The chunk-300 run showed answer_correctness 0.40, answer_groundedness 0.70, retrieval_relevance 0.86, and latency 3.11s.
- Two traces inspected: Yes inpsectedd all 3.
- Decision: The smaller chunk size was not a clear improvement overall. The candidate result was much better.
- Cost or latency tradeoff: The chunk-300 run was slower on the inspected example, so it had worse latency without a clear quality gain.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.